## 🎯 Learning Objectives
* Design and implement a multi-crew architecture for complex problem-solving.
* Orchestrate asynchronous execution of multiple CrewAI crews.
* Effectively pass outputs from one crew as inputs to another.
* Define agents and tasks suitable for a multi-stage, collaborative workflow.


# ADV03-L05: Exercise: Build an Async Multi-Crew Research Pipeline

## Task Description

In this exercise, you will design and implement an advanced asynchronous multi-crew research pipeline using CrewAI. This pipeline will simulate a common business scenario where initial research informs strategic planning. Your solution should demonstrate the ability to chain multiple crews, with the output of one crew serving as the input for the next, all executed asynchronously.

### Scenario

Imagine a company needs to understand a new emerging technology (e.g., "Quantum Computing in Drug Discovery") and then develop a strategic plan for how to leverage or respond to it.

### Pipeline Structure

1.  **Research Crew (Crew 1)**:
    *   **Goal**: Conduct in-depth research on a given topic and synthesize findings into a comprehensive report.
    *   **Agents**: `Senior Researcher`, `Data Analyst`.
    *   **Tools**: A search tool (you will use a mock tool for this exercise).
    *   **Tasks**: Initial information gathering, detailed analysis, report generation.
    *   **Output**: A detailed research report on the specified topic.

2.  **Strategy Crew (Crew 2)**:
    *   **Goal**: Take the research report from Crew 1 and develop a strategic plan, including potential opportunities, threats, and actionable recommendations.
    *   **Agents**: `Strategic Planner`, `Business Analyst`.
    *   **Tools**: None explicitly required for this crew, as its primary input is the research report.
    *   **Tasks**: Analyze research report, identify strategic implications, formulate a strategic plan.
    *   **Output**: A strategic plan document based on the research findings.

### Requirements

*   **CrewAI Framework**: Utilize the `crewai` library for defining agents, tasks, and crews.
*   **Asynchronous Execution**: Implement the pipeline such that both crews run asynchronously. The `kickoff()` method of each crew should be awaited within an `async` function.
*   **Output Chaining**: The final output of the Research Crew must be passed as a direct input to the Strategy Crew's tasks.
*   **Agent Definitions**: Define at least two agents per crew, each with a distinct `role`, `goal`, and `backstory`.
*   **Task Definitions**: Define at least one task per agent, with clear `description` and `expected_output`.
*   **Tools**: Implement a simple mock search tool for the Research Crew to simulate external data fetching.
*   **Clear Output**: The final output should clearly present both the research report and the strategic plan.

### Evaluation Criteria

*   **Correctness**: Does the code run without errors and produce the expected outputs?
*   **Asynchronous Flow**: Is the asynchronous execution correctly implemented using `asyncio` and `await`?
*   **Information Flow**: Is the output of the first crew successfully passed as input to the second crew?
*   **Agent/Task Quality**: Are the agents and tasks well-defined, reflecting their roles and contributing to the overall goal?
*   **Readability**: Is the code clean, well-commented, and easy to understand?


In [ ]:
# Install necessary libraries (if not already installed)
# !pip install crewai crewai_tools 'openai<1.0.0' --quiet

import os
import asyncio
from crewai import Agent, Task, Crew, Process
from crewai_tools import BaseTool

# --- Environment Setup ---
# Set your OpenAI API key. In a real scenario, use environment variables.
# For this exercise, we'll use a placeholder. Replace with your actual key if you want to use a real LLM.
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

# For local models or other providers, configure accordingly.
# For example, using Ollama:
# os.environ["OPENAI_API_BASE"] = "http://localhost:11434/v1"
# os.environ["OPENAI_MODEL_NAME"] = "ollama/llama3"
# os.environ["OPENAI_API_KEY"] = "sk-12345"

# Using a mock LLM for demonstration purposes to avoid API key requirements
# In a real application, you'd replace this with a proper LLM client.
class MockLLM:
    def __init__(self, model_name="mock-model"):
        self.model_name = model_name

    def chat(self, messages, **kwargs):
        # Simulate a response based on the last message content
        last_message = messages[-1]['content']
        if "research" in last_message.lower() or "analyze" in last_message.lower():
            return type('obj', (object,), {'content': "Mock LLM Research Response: This is a simulated research finding based on your query. Quantum computing shows promise in drug discovery by accelerating molecular simulations and optimizing drug design. Challenges include error correction and hardware limitations. Key players are IBM, Google, and D-Wave. The market is expected to grow significantly by 2030."})
        elif "strategy" in last_message.lower() or "plan" in last_message.lower():
            return type('obj', (object,), {'content': "Mock LLM Strategy Response: Based on the research, a strategic plan could involve: 1. Investing in quantum algorithm R&D. 2. Forming partnerships with quantum hardware providers. 3. Training a specialized workforce. 4. Focusing on early-stage drug discovery applications. Risks include high investment costs and uncertain timelines."})
        else:
            return type('obj', (object,), {'content': "Mock LLM Generic Response: This is a generic response from the mock LLM."})

    def invoke(self, input_text, **kwargs):
        # Simulate a response for invoke method (used by some agents/tasks)
        if "research" in input_text.lower() or "analyze" in input_text.lower():
            return "Mock LLM Research Response: This is a simulated research finding based on your query. Quantum computing shows promise in drug discovery by accelerating molecular simulations and optimizing drug design. Challenges include error correction and hardware limitations. Key players are IBM, Google, and D-Wave. The market is expected to grow significantly by 2030."
        elif "strategy" in input_text.lower() or "plan" in input_text.lower():
            return "Mock LLM Strategy Response: Based on the research, a strategic plan could involve: 1. Investing in quantum algorithm R&D. 2. Forming partnerships with quantum hardware providers. 3. Training a specialized workforce. 4. Focusing on early-stage drug discovery applications. Risks include high investment costs and uncertain timelines."
        else:
            return "Mock LLM Generic Response: This is a generic response from the mock LLM."

# Instantiate the mock LLM
mock_llm = MockLLM()

# --- Mock Search Tool Definition ---
class MockSearchTool(BaseTool):
    name: str = "Mock Search Tool"
    description: str = "A tool that simulates searching the web for information on a given query."

    def _run(self, query: str) -> str:
        # Simulate different search results based on query keywords
        if "quantum computing drug discovery" in query.lower():
            return (
                "Recent advancements in quantum computing are revolutionizing drug discovery by enabling faster and more accurate simulations of molecular interactions. Companies like IBM and Google are investing heavily. Challenges include qubit stability and error correction. The market is projected to reach $X billion by 2030, with applications in lead optimization and de novo drug design. Key research areas include quantum chemistry and machine learning integration."
            )
        elif "market trends" in query.lower():
            return (
                "The market for quantum computing in healthcare is experiencing rapid growth, driven by pharmaceutical R&D. Major trends include increased venture capital funding, academic-industry collaborations, and the development of specialized quantum algorithms for biological problems. Regulatory frameworks are still nascent."
            )
        elif "competitors" in query.lower():
            return (
                "Leading competitors in quantum drug discovery include pharmaceutical giants partnering with quantum hardware providers (e.g., Pfizer with IBM Quantum), specialized startups (e.g., Entropica Labs, QC Ware), and academic research institutions. Each focuses on different aspects, from algorithm development to specific therapeutic areas."
            )
        else:
            return f"No specific mock data for '{query}'. General search result: Quantum computing is a rapidly evolving field with potential across many industries, including drug discovery. Further research is needed."

# Instantiate the mock search tool
mock_search_tool = MockSearchTool()

print("Setup complete. Mock LLM and Mock Search Tool are ready.")


## Your Turn! Implement the Multi-Crew Asynchronous Research Pipeline

Now it's your turn to build the two crews and orchestrate their asynchronous execution. Remember to:

1.  Define the `Research Crew` with `Senior Researcher` and `Data Analyst` agents, using the `MockSearchTool`.
2.  Define the `Strategy Crew` with `Strategic Planner` and `Business Analyst` agents.
3.  Ensure the output of the `Research Crew` is passed as input to the `Strategy Crew`.
4.  Execute the entire pipeline asynchronously using `asyncio`.

Feel free to choose a specific topic for the research, e.g., "The impact of AI on personalized medicine in 2026" or "Blockchain applications in supply chain logistics by 2028".


In [ ]:
import os
import asyncio
from crewai import Agent, Task, Crew, Process
from crewai_tools import BaseTool

# Re-using the MockLLM and MockSearchTool from the setup cell
# In a real scenario, you would import them or ensure they are defined globally.

# --- Mock LLM Definition (for completeness, assuming it's not globally available) ---
class MockLLM:
    def __init__(self, model_name="mock-model"):
        self.model_name = model_name

    def chat(self, messages, **kwargs):
        last_message = messages[-1]['content']
        if "research" in last_message.lower() or "analyze" in last_message.lower():
            return type('obj', (object,), {'content': "Mock LLM Research Response: This is a simulated research finding based on your query. Quantum computing shows promise in drug discovery by accelerating molecular simulations and optimizing drug design. Challenges include error correction and hardware limitations. Key players are IBM, Google, and D-Wave. The market is expected to grow significantly by 2030."})
        elif "strategy" in last_message.lower() or "plan" in last_message.lower():
            return type('obj', (object,), {'content': "Mock LLM Strategy Response: Based on the research, a strategic plan could involve: 1. Investing in quantum algorithm R&D. 2. Forming partnerships with quantum hardware providers. 3. Training a specialized workforce. 4. Focusing on early-stage drug discovery applications. Risks include high investment costs and uncertain timelines."})
        else:
            return type('obj', (object,), {'content': "Mock LLM Generic Response: This is a generic response from the mock LLM."})

    def invoke(self, input_text, **kwargs):
        if "research" in input_text.lower() or "analyze" in input_text.lower():
            return "Mock LLM Research Response: This is a simulated research finding based on your query. Quantum computing shows promise in drug discovery by accelerating molecular simulations and optimizing drug design. Challenges include error correction and hardware limitations. Key players are IBM, Google, and D-Wave. The market is expected to grow significantly by 2030."
        elif "strategy" in input_text.lower() or "plan" in input_text.lower():
            return "Mock LLM Strategy Response: Based on the research, a strategic plan could involve: 1. Investing in quantum algorithm R&D. 2. Forming partnerships with quantum hardware providers. 3. Training a specialized workforce. 4. Focusing on early-stage drug discovery applications. Risks include high investment costs and uncertain timelines."
        else:
            return "Mock LLM Generic Response: This is a generic response from the mock LLM."

mock_llm = MockLLM()

# --- Mock Search Tool Definition (for completeness) ---
class MockSearchTool(BaseTool):
    name: str = "Mock Search Tool"
    description: str = "A tool that simulates searching the web for information on a given query."

    def _run(self, query: str) -> str:
        if "quantum computing drug discovery" in query.lower():
            return (
                "Recent advancements in quantum computing are revolutionizing drug discovery by enabling faster and more accurate simulations of molecular interactions. Companies like IBM and Google are investing heavily. Challenges include qubit stability and error correction. The market is projected to reach $X billion by 2030, with applications in lead optimization and de novo drug design. Key research areas include quantum chemistry and machine learning integration."
            )
        elif "market trends" in query.lower():
            return (
                "The market for quantum computing in healthcare is experiencing rapid growth, driven by pharmaceutical R&D. Major trends include increased venture capital funding, academic-industry collaborations, and the development of specialized quantum algorithms for biological problems. Regulatory frameworks are still nascent."
            )
        elif "competitors" in query.lower():
            return (
                "Leading competitors in quantum drug discovery include pharmaceutical giants partnering with quantum hardware providers (e.g., Pfizer with IBM Quantum), specialized startups (e.g., Entropica Labs, QC Ware), and academic research institutions. Each focuses on different aspects, from algorithm development to specific therapeutic areas."
            )
        else:
            return f"No specific mock data for '{query}'. General search result: Quantum computing is a rapidly evolving field with potential across many industries, including drug discovery. Further research is needed."

mock_search_tool = MockSearchTool()

# --- Define Agents for Research Crew ---
researcher = Agent(
    role='Senior Researcher',
    goal='Conduct comprehensive research on emerging technologies and market trends.',
    backstory="An experienced researcher with a knack for uncovering critical information and synthesizing complex data points.",
    verbose=True,
    allow_delegation=False,
    llm=mock_llm # Use the mock LLM
)

data_analyst = Agent(
    role='Data Analyst',
    goal='Analyze research findings and extract key insights and statistics.',
    backstory="A meticulous data analyst who excels at identifying patterns, trends, and crucial data points from raw information.",
    verbose=True,
    allow_delegation=True,
    llm=mock_llm # Use the mock LLM
)

# --- Define Tasks for Research Crew ---
research_topic = "Quantum Computing in Drug Discovery by 2026"

search_task = Task(
    description=f"Search for the latest advancements, market trends, key players, and challenges in {research_topic}. Focus on practical applications and future outlook.",
    expected_output="A detailed summary of search results, including URLs (if real search) or key findings from the mock tool.",
    tools=[mock_search_tool],
    agent=researcher
)

analyze_research_task = Task(
    description="Analyze the search results provided by the Senior Researcher. Identify key opportunities, threats, and significant developments. Synthesize this into a concise report.",
    expected_output="A comprehensive research report (2-3 paragraphs) detailing the current state, future potential, and challenges of quantum computing in drug discovery.",
    agent=data_analyst,
    context=[search_task] # Data Analyst uses the output of the search task
)

# --- Define Research Crew ---
research_crew = Crew(
    agents=[researcher, data_analyst],
    tasks=[search_task, analyze_research_task],
    process=Process.sequential, # Tasks run sequentially
    verbose=2 # Shows more details about the crew's execution
)

# --- Define Agents for Strategy Crew ---
strategist = Agent(
    role='Strategic Planner',
    goal='Develop actionable strategic recommendations based on research findings.',
    backstory="A visionary strategist with a proven track record of transforming insights into executable business plans.",
    verbose=True,
    allow_delegation=False,
    llm=mock_llm # Use the mock LLM
)

business_analyst = Agent(
    role='Business Analyst',
    goal='Evaluate strategic options and outline implementation steps.',
    backstory="An astute business analyst focused on practical implementation and assessing the feasibility and impact of strategic initiatives.",
    verbose=True,
    allow_delegation=True,
    llm=mock_llm # Use the mock LLM
)

# --- Define Tasks for Strategy Crew ---
analyze_report_task = Task(
    description="Analyze the provided research report on quantum computing in drug discovery. Identify key strategic implications, opportunities, and potential risks for a pharmaceutical company.",
    expected_output="A summary of strategic implications derived from the research report.",
    agent=strategist
    # Context will be provided dynamically from research_crew_output
)

formulate_strategy_task = Task(
    description="Based on the strategic implications, formulate a concise strategic plan (3-4 key action points) for a pharmaceutical company to leverage or respond to quantum computing advancements. Include potential timelines and resource considerations.",
    expected_output="A strategic plan document with actionable recommendations, timelines, and resource considerations.",
    agent=business_analyst,
    context=[analyze_report_task] # Business Analyst uses the output of the strategist's analysis
)

# --- Define Strategy Crew (will be initialized with dynamic input) ---
# We'll initialize it inside the async function to pass the research_crew_output

# --- Asynchronous Multi-Crew Orchestration ---
async def run_multi_crew_pipeline(topic: str):
    print(f"\n--- Starting Research for: {topic} ---")
    # Update research topic for the first task dynamically if needed
    search_task.description = f"Search for the latest advancements, market trends, key players, and challenges in {topic}. Focus on practical applications and future outlook."
    analyze_research_task.description = f"Analyze the search results provided by the Senior Researcher. Identify key opportunities, threats, and significant developments. Synthesize this into a concise report on {topic}."

    # Kickoff the Research Crew asynchronously
    research_crew_output = await research_crew.kickoff()
    print("\n--- Research Crew Finished ---")
    print("Research Report:\n", research_crew_output)

    print(f"\n--- Starting Strategy Development based on Research ---")

    # Pass the output of the Research Crew as input to the Strategy Crew's tasks
    # This is done by setting the 'inputs' for the strategy crew's tasks.
    # For this example, we'll pass it directly to the first task's description/context.
    # In a more complex scenario, you might parse research_crew_output into specific inputs.

    # Update strategy tasks with the research report as context
    analyze_report_task.description = f"Analyze the following research report on {topic}:\n\n{research_crew_output}\n\nIdentify key strategic implications, opportunities, and potential risks for a pharmaceutical company."
    formulate_strategy_task.description = f"Based on the strategic implications derived from the research report on {topic} (provided in the previous task), formulate a concise strategic plan (3-4 key action points) for a pharmaceutical company to leverage or respond to these advancements. Include potential timelines and resource considerations."

    strategy_crew = Crew(
        agents=[strategist, business_analyst],
        tasks=[analyze_report_task, formulate_strategy_task],
        process=Process.sequential,
        verbose=2,
        # inputs={'research_report': research_crew_output} # Can also pass inputs this way if tasks are designed to consume them
    )

    # Kickoff the Strategy Crew asynchronously
    strategy_crew_output = await strategy_crew.kickoff()
    print("\n--- Strategy Crew Finished ---")
    print("Strategic Plan:\n", strategy_crew_output)

    return research_crew_output, strategy_crew_output

# --- Run the pipeline ---
if __name__ == "__main__":
    # Define the topic for the pipeline
    pipeline_topic = "The Future of Personalized Medicine with AI by 2028"

    # Run the asynchronous pipeline
    final_research_report, final_strategic_plan = asyncio.run(run_multi_crew_pipeline(pipeline_topic))

    print("\n==================================================")
    print("FINAL PIPELINE RESULTS")
    print("==================================================")
    print("\n--- Comprehensive Research Report ---")
    print(final_research_report)
    print("\n--- Strategic Action Plan ---")
    print(final_strategic_plan)
    print("==================================================")
